In [1]:
import os
import pandas as pd
import random

In [2]:
# === CONFIG ===
INPUT_CSVS = [
    "rag_outputs_BM25_T5_per_query_3000.csv",
    "rag_outputs_BM25_BART_per_query_3000.csv",
    "rag_outputs_dpr_BART_per_query_3000.csv",
    "rag_outputs_dpr_T5_per_query_3000.csv",
]

OUTPUT_CSV = "human_eval_rag.csv"

N_SAMPLE = 40
SEED = 42

In [3]:
KEEP_COLS = [
    "idx", "question", "gold", "pred",
    "em", "f1", "rougeL", "bleu",
]


In [4]:
def read_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if "idx" not in df.columns:
        raise ValueError(f"'idx' column missing in {path}")
    return df


In [5]:
random.seed(SEED)

master_df = read_csv(INPUT_CSVS[0])

if N_SAMPLE > len(master_df):
    N_SAMPLE = len(master_df)

sample_df = master_df.sample(n=N_SAMPLE, random_state=SEED)
sample_idx = set(sample_df["idx"].tolist())

print(f"Sampled {len(sample_idx)} questions")


Sampled 40 questions


In [6]:
rows_out = []

for path in INPUT_CSVS:
    df = read_csv(path)

    df_s = df[df["idx"].isin(sample_idx)].copy()

    cols = [c for c in KEEP_COLS if c in df_s.columns]
    df_s = df_s[cols]

    tag = os.path.splitext(os.path.basename(path))[0]
    df_s["source"] = tag

    rows_out.append(df_s)

    missing = len(sample_idx) - df_s["idx"].nunique()
    if missing > 0:
        print(f"[WARN] {tag}: missing {missing} questions")


In [7]:
out_df = pd.concat(rows_out, ignore_index=True)

# Felder für dich (manuelle Bewertung)
out_df["human_correctness_0_2"] = ""
out_df["human_faithfulness_0_2"] = ""
out_df["human_notes"] = ""

out_df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)
print("Rows:", len(out_df))
print("Unique questions:", out_df["idx"].nunique())


Saved: human_eval_rag.csv
Rows: 160
Unique questions: 40
